<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
Vector DB로 유사 문장 검색 개념 이해
</div>

# 기본환경 설정

In [1]:
%%capture
%pip install -q -U langchain-core langchain_huggingface chromadb

# 임베딩 생성기 (한국어 포함 다국어 모델)

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

MODEL_EMBED = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2" # intfloat/multilingual-e5-base

# Chroma 벡터 저장소 이해

Chroma는 기본적으로 **L2(유클리드) 거리**를 사용합니다 — 거리가 작을수록 의미가 가깝습니다.

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

In [4]:
# 임베딩 모델 로드 (다국어 지원)
embed_model = SentenceTransformer(MODEL_EMBED)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# 데이터셋: 다국어 문장들
sentences = [
    "고양이가 소파 위에서 자고 있다.",        # 한국어
    "The cat is sleeping on the couch.",      # 영어
    "Le chat dort sur le canapé.",            # 프랑스어
    "Le chien dort sur le canapé.",           # 프랑스어 : 개가 소파에서 자고 있어요.
    "Die Katze schläft auf dem Sofa.",        # 독일어
    "Der Hund schläft auf dem Sofa.",         # 독일어 : 개가 소파에서 자고 있어요.
    "El gato duerme en el sofá.",             # 스페인어
    "El caballo está corriendo por el prado." # 스페인어 : 말이 초원을 달리고 있어요.
]

In [ ]:
# vector db 저장
sentence_embeddings = embed_model.encode(sentences, convert_to_numpy=True)

client = chromadb.Client()  # 인메모리 클라이언트
collection = client.get_or_create_collection(name="embed_test")  # 기본 거리: L2

collection.add(
    ids=[f"sent-{i}" for i in range(len(sentences))],
    embeddings=sentence_embeddings.tolist(),
    documents=sentences,
)

print(f"저장된 문장 수: {collection.count()}")

In [ ]:
len(sentence_embeddings[0])

In [ ]:
# query 함수
def queryVector(query, result_cnt=3):
    query_embedding = embed_model.encode([query], convert_to_numpy=True).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=result_cnt)

    print("\nTop Matches:")
    for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
        print(f"{i+1}. {doc} (거리: {dist:.4f})")

In [ ]:
# 고양이 질문
queryVector("소파에 있는 고양이", 3)

In [ ]:
# 강아지 질문
queryVector("소파에 있는 강아지", 3)

In [ ]:
# 초원 질문
queryVector("초원에 석양이 지고 있어요.", 3)